<a href="https://colab.research.google.com/github/hongyuw0427/Final_Year_Project/blob/model-only/FYP_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


# FE+EMBEDDINGS Part

In [ ]:
# ==========================================================
# SCRIPT 4: FEATURE EXTRACTION EXPERIMENTS (AFTER BASELINE)
# - TF-IDF (sparse)
# - Sentence Embeddings (all-MiniLM-L6-v2)
# - Fusion = hstack(TF-IDF sparse + Embeddings as CSR sparse)
# - Models: LR / RF / SVM (LinearSVC + CalibratedClassifierCV)
# - 4 splits: 90/10, 80/20, 70/30, 60/40
# ==========================================================

import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

from scipy.sparse import hstack, csr_matrix

import matplotlib.pyplot as plt
import seaborn as sns

# ----------------------------
# Config
# ----------------------------
DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/FYP/cyberbullying_tweets_final_cleaned.csv"
OUT_DIR   = "/content/drive/MyDrive/Colab Notebooks/FYP/FE+EMB_result"
os.makedirs(OUT_DIR, exist_ok=True)

SAVE_RESULTS_PATH = os.path.join(OUT_DIR, "FE+EMB_result.csv")
CM_DIR = os.path.join(OUT_DIR, "confusion_matrices")
os.makedirs(CM_DIR, exist_ok=True)

SPLITS = [(0.9,0.1),(0.8,0.2),(0.7,0.3),(0.6,0.4)]
SEED = 42

SAVE_CM_IMAGES = True
SHOW_CM = False  # True if you want to show inline

# ----------------------------
# Confusion Matrix (image)
# ----------------------------
def save_confusion_matrix_image(y_true, y_pred, labels, title, save_path):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels)
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    if SHOW_CM:
        plt.show()
    plt.close()

# ----------------------------
# Load data
# ----------------------------
df = pd.read_csv(DATA_PATH)
df["clean_text"] = df["clean_text"].fillna("").astype(str)
df["cyberbullying_type"] = df["cyberbullying_type"].astype(str)

texts = df["clean_text"].tolist()
labels = df["cyberbullying_type"].tolist()

le = LabelEncoder()
y = le.fit_transform(labels)
label_names = le.classes_
num_classes = len(label_names)

print("Loaded:", df.shape)
print("Classes:", label_names)
print("\nClass distribution:")
print(df["cyberbullying_type"].value_counts())

# ==========================================================
# Embeddings setup (compute once)
# ==========================================================
print("\n=================================")
print(" Sentence Embeddings Extraction")
print("=================================\n")

embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("Computing embeddings for all texts (one-time)...")
X_embed_all = embedder.encode(texts, convert_to_numpy=True, show_progress_bar=True)
print("Embeddings shape:", X_embed_all.shape)

# ==========================================================
# Run 4 splits and train models for:
# - TFIDF
# - EMB
# - FUSION
# ==========================================================
results = []

def train_eval_block(model_dict, Xtr, Xte, y_train, y_test, y_test_bin, train_frac, block_name):
    for name, model in model_dict.items():
        print(f"\n🚀 Training {name} on {block_name} ({int(train_frac*100)}% train)...")

        model.fit(Xtr, y_train)
        y_pred = model.predict(Xte)

        # ROC-AUC needs proba
        if hasattr(model, "predict_proba"):
            y_proba = model.predict_proba(Xte)
            roc = roc_auc_score(y_test_bin, y_proba, average="macro", multi_class="ovr")
        else:
            y_proba = None
            roc = np.nan

        results.append({
            "model": name,
            "train_frac": train_frac,
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred, average="macro"),
            "recall": recall_score(y_test, y_pred, average="macro"),
            "f1": f1_score(y_test, y_pred, average="macro"),
            "roc_auc": roc
        })

        # Confusion matrix image
        if SAVE_CM_IMAGES:
            cm_name = f"CM_{name}_{int(train_frac*100)}.png"
            cm_path = os.path.join(CM_DIR, cm_name)
            save_confusion_matrix_image(
                y_true=y_test,
                y_pred=y_pred,
                labels=label_names,
                title=f"{name} ({int(train_frac*100)}% train) - {block_name}",
                save_path=cm_path
            )

        print(f"✅ Done {name} | Acc={results[-1]['accuracy']:.4f} F1={results[-1]['f1']:.4f}")

for train_frac, test_frac in SPLITS:
    split_tag = f"{int(train_frac*100)}_{int(test_frac*100)}"

    print("\n" + "="*90)
    print(f"RUN SPLIT: {split_tag}")
    print("="*90)

    idx = np.arange(len(texts))
    idx_train, idx_test, y_train, y_test = train_test_split(
        idx, y, test_size=test_frac, random_state=SEED, stratify=y
    )

    X_train_text = [texts[i] for i in idx_train]
    X_test_text  = [texts[i] for i in idx_test]

    y_test_bin = label_binarize(y_test, classes=range(num_classes))

    # ----------------------------
    # A) TF-IDF features (fit on TRAIN only to avoid leakage)
    # ----------------------------
    print("\n[TF-IDF] Fitting on training set and transforming...")
    tfidf = TfidfVectorizer(
        max_features=3000,
        ngram_range=(1,2),
        stop_words="english"
    )
    X_train_tfidf = tfidf.fit_transform(X_train_text)
    X_test_tfidf  = tfidf.transform(X_test_text)
    print("[TF-IDF] Done. Shapes:", X_train_tfidf.shape, X_test_tfidf.shape)

    # ----------------------------
    # B) Embedding features (dense)
    # ----------------------------
    print("\n[EMB] Preparing embeddings split...")
    X_train_emb = X_embed_all[idx_train]
    X_test_emb  = X_embed_all[idx_test]
    print("[EMB] Done. Shapes:", X_train_emb.shape, X_test_emb.shape)

    # ----------------------------
    # C) Fusion features (sparse hstack)
    # ----------------------------
    print("\n[FUSION] Building sparse fusion via hstack...")
    X_train_emb_sp = csr_matrix(X_train_emb)
    X_test_emb_sp  = csr_matrix(X_test_emb)
    X_train_fusion = hstack([X_train_tfidf, X_train_emb_sp], format="csr")
    X_test_fusion  = hstack([X_test_tfidf,  X_test_emb_sp],  format="csr")
    print("[FUSION] Done. Shapes:", X_train_fusion.shape, X_test_fusion.shape)

    # Models
    models_tfidf = {
        "LR_TFIDF": LogisticRegression(max_iter=2000),
        "RF_TFIDF": RandomForestClassifier(n_jobs=-1),
        "SVM_TFIDF": CalibratedClassifierCV(LinearSVC(dual=False), cv=3),
    }

    models_emb = {
        "LR_EMB": LogisticRegression(max_iter=2000),
        "RF_EMB": RandomForestClassifier(n_jobs=-1),
        "SVM_EMB": CalibratedClassifierCV(LinearSVC(dual=False), cv=3),
    }

    models_fusion = {
        "LR_FUSION": LogisticRegression(max_iter=2000),
        "RF_FUSION": RandomForestClassifier(n_jobs=-1),
        "SVM_FUSION": CalibratedClassifierCV(LinearSVC(dual=False), cv=3),
    }

    # Run blocks
    train_eval_block(models_tfidf, X_train_tfidf, X_test_tfidf, y_train, y_test, y_test_bin, train_frac, "TF-IDF")
    train_eval_block(models_emb, X_train_emb, X_test_emb, y_train, y_test, y_test_bin, train_frac, "EMB")
    train_eval_block(models_fusion, X_train_fusion, X_test_fusion, y_train, y_test, y_test_bin, train_frac, "FUSION")

# ----------------------------
# Save results
# ----------------------------
results_df = pd.DataFrame(results)
results_df.to_csv(SAVE_RESULTS_PATH, index=False)

print("\n======================================")
print("ALL DONE ✅")
print("Saved results to:", SAVE_RESULTS_PATH)
if SAVE_CM_IMAGES:
    print("Saved confusion matrices to:", CM_DIR)
print("======================================\n")

results_df

Loaded: (45736, 3)
Classes: ['age' 'ethnicity' 'gender' 'not_cyberbullying' 'other_cyberbullying'
 'religion']

Class distribution:
cyberbullying_type
age                    7954
ethnicity              7847
religion               7698
gender                 7570
not_cyberbullying      7377
other_cyberbullying    7290
Name: count, dtype: int64

 Sentence Embeddings Extraction

Computing embeddings for all texts (one-time)...


Batches:   0%|          | 0/1430 [00:00<?, ?it/s]

Embeddings shape: (45736, 384)

RUN SPLIT: 90_10

[TF-IDF] Fitting on training set and transforming...
[TF-IDF] Done. Shapes: (41162, 3000) (4574, 3000)

[EMB] Preparing embeddings split...
[EMB] Done. Shapes: (41162, 384) (4574, 384)

[FUSION] Building sparse fusion via hstack...
[FUSION] Done. Shapes: (41162, 3384) (4574, 3384)

🚀 Training LR_TFIDF on TF-IDF (90% train)...
✅ Done LR_TFIDF | Acc=0.8124 F1=0.8109

🚀 Training RF_TFIDF on TF-IDF (90% train)...
✅ Done RF_TFIDF | Acc=0.7995 F1=0.7973

🚀 Training SVM_TFIDF on TF-IDF (90% train)...
✅ Done SVM_TFIDF | Acc=0.8150 F1=0.8123

🚀 Training LR_EMB on EMB (90% train)...
✅ Done LR_EMB | Acc=0.8080 F1=0.8039

🚀 Training RF_EMB on EMB (90% train)...
✅ Done RF_EMB | Acc=0.7599 F1=0.7548

🚀 Training SVM_EMB on EMB (90% train)...
✅ Done SVM_EMB | Acc=0.8085 F1=0.8030

🚀 Training LR_FUSION on FUSION (90% train)...
✅ Done LR_FUSION | Acc=0.8194 F1=0.8160

🚀 Training RF_FUSION on FUSION (90% train)...
✅ Done RF_FUSION | Acc=0.7704 F1=0.7679



,model,train_frac,accuracy,precision,recall,f1,roc_auc
0,LR_TFIDF,0.9,0.812418,0.815671,0.807714,0.810850,0.962481
1,RF_TFIDF,0.9,0.799519,0.802412,0.794010,0.797322,0.952791
2,SVM_TFIDF,0.9,0.815042,0.816402,0.810262,0.812290,0.961397
3,LR_EMB,0.9,0.808045,0.804912,0.803359,0.803916,0.962378
4,RF_EMB,0.9,0.759948,0.757250,0.753770,0.754759,0.932787
5,SVM_EMB,0.9,0.808483,0.803215,0.803614,0.803019,0.960459
6,LR_FUSION,0.9,0.819414,0.818549,0.814716,0.815993,0.967158
7,RF_FUSION,0.9,0.770442,0.773591,0.764355,0.767872,0.936274
8,SVM_FUSION,0.9,0.822475,0.821180,0.817764,0.818731,0.965006
9,LR_TFIDF,0.8,0.814058,0.813297,0.809157,0.810865,0.963247


# Co-Relationship analysis

In [ ]:
# ============================================
# FULL CORRELATION ANALYSIS
# other_cyberbullying vs not_cyberbullying
# ============================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression

from sentence_transformers import SentenceTransformer, util
from collections import Counter

# --------------------------------------------------
# 1. Load dataset
# --------------------------------------------------
DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/FYP/cyberbullying_tweets_final_cleaned.csv"

df = pd.read_csv(DATA_PATH)

# Filter only the two classes
df_sub = df[df["cyberbullying_type"].isin(
    ["other_cyberbullying", "not_cyberbullying"]
)].copy()

df_sub.reset_index(drop=True, inplace=True)

texts = df_sub["clean_text"].astype(str).tolist()
labels = df_sub["cyberbullying_type"].tolist()

print("Dataset shape:", df_sub.shape)
print("\nClass distribution:")
print(df_sub["cyberbullying_type"].value_counts(), "\n")

# Encode labels
le = LabelEncoder()
y = le.fit_transform(labels)
label_names = le.classes_

# --------------------------------------------------
# 2. Baseline classifier (TF-IDF + LR) for confusion
# --------------------------------------------------
X_train_text, X_test_text, y_train, y_test = train_test_split(
    texts, y, test_size=0.2, stratify=y, random_state=42
)

tfidf = TfidfVectorizer(max_features=5000)
X_train = tfidf.fit_transform(X_train_text)
X_test = tfidf.transform(X_test_text)

clf = LogisticRegression(max_iter=2000)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)

print("Confusion Matrix (other vs not):")
cm = confusion_matrix(y_test, y_pred)
print(pd.DataFrame(cm, index=label_names, columns=label_names), "\n")

# --------------------------------------------------
# 3. TF-IDF cosine similarity (class centroids)
# --------------------------------------------------
print("Computing TF-IDF similarity...")

other_texts = df_sub[df_sub["cyberbullying_type"]=="other_cyberbullying"]["clean_text"].tolist()
not_texts   = df_sub[df_sub["cyberbullying_type"]=="not_cyberbullying"]["clean_text"].tolist()

tfidf_all = TfidfVectorizer(max_features=5000)
tfidf_matrix = tfidf_all.fit_transform(other_texts + not_texts)

o = tfidf_matrix[:len(other_texts)]
n = tfidf_matrix[len(other_texts):]

o_mean = np.asarray(o.mean(axis=0))
n_mean = np.asarray(n.mean(axis=0))

tfidf_sim = cosine_similarity(o_mean, n_mean)[0][0]
print("TF-IDF Average Similarity:", round(tfidf_sim, 4), "\n")

# --------------------------------------------------
# 4. Sentence embedding similarity (MiniLM)
# --------------------------------------------------
print("Computing sentence embedding similarity...")

embedder = SentenceTransformer("all-MiniLM-L6-v2")

embed_other = embedder.encode(other_texts, convert_to_numpy=True, show_progress_bar=True)
embed_not   = embedder.encode(not_texts,   convert_to_numpy=True, show_progress_bar=True)

# Mean embedding similarity
emb_sim = util.cos_sim(
    embed_other.mean(axis=0),
    embed_not.mean(axis=0)
).item()

print("Embedding Semantic Similarity (mean):", round(emb_sim, 4))

# Distribution-level similarity
pair_sim = cosine_similarity(embed_other, embed_not)
print("\nEmbedding similarity distribution:")
print("Mean:", round(pair_sim.mean(), 4))
print("Median:", round(np.median(pair_sim), 4))
print("90th percentile:", round(np.percentile(pair_sim, 90), 4), "\n")

# --------------------------------------------------
# 5. Lexical overlap (top keywords)
# --------------------------------------------------
def top_words(texts, n=30):
    words = " ".join(texts).split()
    return set([w for w,_ in Counter(words).most_common(n)])

top_other = top_words(other_texts)
top_not   = top_words(not_texts)

overlap_ratio = len(top_other & top_not) / len(top_other | top_not)
print("Top-word overlap ratio:", round(overlap_ratio, 4), "\n")

# --------------------------------------------------
# 6. Model confidence overlap
# --------------------------------------------------
conf_df = pd.DataFrame({
    "true": le.inverse_transform(y_test),
    "confidence": y_prob.max(axis=1)
})

print("Average prediction confidence by class:")
print(conf_df.groupby("true")["confidence"].mean(), "\n")

# --------------------------------------------------
# 7. Misclassified examples
# --------------------------------------------------
mis = pd.DataFrame({
    "text": X_test_text,
    "true": le.inverse_transform(y_test),
    "pred": le.inverse_transform(y_pred)
})

mis_filtered = mis[
    (
        (mis["true"]=="other_cyberbullying") &
        (mis["pred"]=="not_cyberbullying")
    )
    |
    (
        (mis["true"]=="not_cyberbullying") &
        (mis["pred"]=="other_cyberbullying")
    )
]

print("Examples of confusion between the two classes:\n")
print(mis_filtered.head(20).to_string(index=False), "\n")

# --------------------------------------------------
# 8. Interpretation
# --------------------------------------------------
print("===== INTERPRETATION =====")

if emb_sim > 0.7:
    print("➡ Very high semantic similarity (>0.70).")
    print("   These two classes are inherently overlapping.")
elif emb_sim > 0.5:
    print("➡ Moderate semantic similarity (0.50–0.70).")
    print("   Model confusion is expected.")
else:
    print("➡ Low semantic similarity (<0.50).")
    print("   Confusion likely due to model limitations.")



Dataset shape: (14667, 3)

Class distribution:
cyberbullying_type
not_cyberbullying      7377
other_cyberbullying    7290
Name: count, dtype: int64 

Confusion Matrix (other vs not):
                     not_cyberbullying  other_cyberbullying
not_cyberbullying                  954                  522
other_cyberbullying                452                 1006 

Computing TF-IDF similarity...
TF-IDF Average Similarity: 0.8438 

Computing sentence embedding similarity...


Batches:   0%|          | 0/228 [00:00<?, ?it/s]

Batches:   0%|          | 0/231 [00:00<?, ?it/s]

Embedding Semantic Similarity (mean): 0.9671

Embedding similarity distribution:
Mean: 0.1214
Median: 0.1143
90th percentile: 0.2312 

Top-word overlap ratio: 0.5385 

Average prediction confidence by class:
true
not_cyberbullying      0.676878
other_cyberbullying    0.645403
Name: confidence, dtype: float64 

Examples of confusion between the two classes:

                                                                                                                 text                true                pred
                                                depends define meaningful people listen v there huge difference voice   not_cyberbullying other_cyberbullying
                                  explain doctor optional surgery planning cant happen next month bcz time recuperate   not_cyberbullying other_cyberbullying
                                                                                              oh really like meow mix other_cyberbullying   not_cyberbullying
        

# Remove Overlap row (other_cyberbullying)

In [ ]:
# ==========================================
# REMOVE OVERLAPPING CLASS:
# other_cyberbullying
# ==========================================

import pandas as pd

# ----------------------------
# Load dataset (after cleaning / outlier removal)
# ----------------------------
DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/FYP/cyberbullying_tweets_final_cleaned.csv"
df = pd.read_csv(DATA_PATH)

print("Original dataset shape:", df.shape)

# ----------------------------
# Class distribution BEFORE
# ----------------------------
print("\nClass distribution BEFORE removal:")
print(df["cyberbullying_type"].value_counts())

# ----------------------------
# Count rows to be removed
# ----------------------------
removed_count = (df["cyberbullying_type"] == "other_cyberbullying").sum()
print("\nRows to be removed (other_cyberbullying):", removed_count)

# ----------------------------
# Drop overlapping class
# ----------------------------
df_no_other = df[df["cyberbullying_type"] != "other_cyberbullying"].copy()

print("\nDataset shape AFTER removal:", df_no_other.shape)

# ----------------------------
# Class distribution AFTER
# ----------------------------
print("\nClass distribution AFTER removal:")
print(df_no_other["cyberbullying_type"].value_counts())

# ----------------------------
# Save final dataset
# ----------------------------
SAVE_PATH = "/content/drive/MyDrive/Colab Notebooks/FYP/cyberbullying_cleaned_dropOverlappedRows.csv"
df_no_other.to_csv(SAVE_PATH, index=False)

print("\nSaved cleaned dataset to:")
print(SAVE_PATH)


Original dataset shape: (45736, 3)

Class distribution BEFORE removal:
cyberbullying_type
age                    7954
ethnicity              7847
religion               7698
gender                 7570
not_cyberbullying      7377
other_cyberbullying    7290
Name: count, dtype: int64

Rows to be removed (other_cyberbullying): 7290

Dataset shape AFTER removal: (38446, 3)

Class distribution AFTER removal:
cyberbullying_type
age                  7954
ethnicity            7847
religion             7698
gender               7570
not_cyberbullying    7377
Name: count, dtype: int64

Saved cleaned dataset to:
/content/drive/MyDrive/Colab Notebooks/FYP/cyberbullying_cleaned_dropOverlappedRows.csv


# Rerun FE+EMBEDDINGS Part after dropped Overlap Rows

In [ ]:
# ==========================================================
# SCRIPT 4: FEATURE EXTRACTION EXPERIMENTS (AFTER BASELINE)
# - TF-IDF (sparse)
# - Sentence Embeddings (all-MiniLM-L6-v2)
# - Fusion = hstack(TF-IDF sparse + Embeddings as CSR sparse)
# - Models: LR / RF / SVM (LinearSVC + CalibratedClassifierCV)
# - 4 splits: 90/10, 80/20, 70/30, 60/40
# ==========================================================

import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

from scipy.sparse import hstack, csr_matrix

import matplotlib.pyplot as plt
import seaborn as sns

# ----------------------------
# Config
# ----------------------------
DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/FYP/cyberbullying_cleaned_dropOverlappedRows.csv"
OUT_DIR   = "/content/drive/MyDrive/Colab Notebooks/FYP/feature_extraction_droppedOverlap"
os.makedirs(OUT_DIR, exist_ok=True)

SAVE_RESULTS_PATH = os.path.join(OUT_DIR, "feature_extraction_droppedOverlap.csv")
CM_DIR = os.path.join(OUT_DIR, "confusion_matrices")
os.makedirs(CM_DIR, exist_ok=True)

SPLITS = [(0.9,0.1),(0.8,0.2),(0.7,0.3),(0.6,0.4)]
SEED = 42

SAVE_CM_IMAGES = True
SHOW_CM = False  # True if you want to show inline

# ----------------------------
# Confusion Matrix (image)
# ----------------------------
def save_confusion_matrix_image(y_true, y_pred, labels, title, save_path):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels)
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    if SHOW_CM:
        plt.show()
    plt.close()

# ----------------------------
# Load data
# ----------------------------
df = pd.read_csv(DATA_PATH)
df["clean_text"] = df["clean_text"].fillna("").astype(str)
df["cyberbullying_type"] = df["cyberbullying_type"].astype(str)

texts = df["clean_text"].tolist()
labels = df["cyberbullying_type"].tolist()

le = LabelEncoder()
y = le.fit_transform(labels)
label_names = le.classes_
num_classes = len(label_names)

print("Loaded:", df.shape)
print("Classes:", label_names)
print("\nClass distribution:")
print(df["cyberbullying_type"].value_counts())

# ==========================================================
# Embeddings setup (compute once)
# ==========================================================
print("\n=================================")
print(" Sentence Embeddings Extraction")
print("=================================\n")

embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("Computing embeddings for all texts (one-time)...")
X_embed_all = embedder.encode(texts, convert_to_numpy=True, show_progress_bar=True)
print("Embeddings shape:", X_embed_all.shape)

# ==========================================================
# Run 4 splits and train models for:
# - TFIDF
# - EMB
# - FUSION
# ==========================================================
results = []

def train_eval_block(model_dict, Xtr, Xte, y_train, y_test, y_test_bin, train_frac, block_name):
    for name, model in model_dict.items():
        print(f"\n🚀 Training {name} on {block_name} ({int(train_frac*100)}% train)...")

        model.fit(Xtr, y_train)
        y_pred = model.predict(Xte)

        # ROC-AUC needs proba
        if hasattr(model, "predict_proba"):
            y_proba = model.predict_proba(Xte)
            roc = roc_auc_score(y_test_bin, y_proba, average="macro", multi_class="ovr")
        else:
            y_proba = None
            roc = np.nan

        results.append({
            "model": name,
            "train_frac": train_frac,
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred, average="macro"),
            "recall": recall_score(y_test, y_pred, average="macro"),
            "f1": f1_score(y_test, y_pred, average="macro"),
            "roc_auc": roc
        })

        # Confusion matrix image
        if SAVE_CM_IMAGES:
            cm_name = f"CM_{name}_{int(train_frac*100)}.png"
            cm_path = os.path.join(CM_DIR, cm_name)
            save_confusion_matrix_image(
                y_true=y_test,
                y_pred=y_pred,
                labels=label_names,
                title=f"{name} ({int(train_frac*100)}% train) - {block_name}",
                save_path=cm_path
            )

        print(f"✅ Done {name} | Acc={results[-1]['accuracy']:.4f} F1={results[-1]['f1']:.4f}")

for train_frac, test_frac in SPLITS:
    split_tag = f"{int(train_frac*100)}_{int(test_frac*100)}"

    print("\n" + "="*90)
    print(f"RUN SPLIT: {split_tag}")
    print("="*90)

    idx = np.arange(len(texts))
    idx_train, idx_test, y_train, y_test = train_test_split(
        idx, y, test_size=test_frac, random_state=SEED, stratify=y
    )

    X_train_text = [texts[i] for i in idx_train]
    X_test_text  = [texts[i] for i in idx_test]

    y_test_bin = label_binarize(y_test, classes=range(num_classes))

    # ----------------------------
    # A) TF-IDF features (fit on TRAIN only to avoid leakage)
    # ----------------------------
    print("\n[TF-IDF] Fitting on training set and transforming...")
    tfidf = TfidfVectorizer(
        max_features=3000,
        ngram_range=(1,2),
        stop_words="english"
    )
    X_train_tfidf = tfidf.fit_transform(X_train_text)
    X_test_tfidf  = tfidf.transform(X_test_text)
    print("[TF-IDF] Done. Shapes:", X_train_tfidf.shape, X_test_tfidf.shape)

    # ----------------------------
    # B) Embedding features (dense)
    # ----------------------------
    print("\n[EMB] Preparing embeddings split...")
    X_train_emb = X_embed_all[idx_train]
    X_test_emb  = X_embed_all[idx_test]
    print("[EMB] Done. Shapes:", X_train_emb.shape, X_test_emb.shape)

    # ----------------------------
    # C) Fusion features (sparse hstack)
    # ----------------------------
    print("\n[FUSION] Building sparse fusion via hstack...")
    X_train_emb_sp = csr_matrix(X_train_emb)
    X_test_emb_sp  = csr_matrix(X_test_emb)
    X_train_fusion = hstack([X_train_tfidf, X_train_emb_sp], format="csr")
    X_test_fusion  = hstack([X_test_tfidf,  X_test_emb_sp],  format="csr")
    print("[FUSION] Done. Shapes:", X_train_fusion.shape, X_test_fusion.shape)

    # Models
    models_tfidf = {
        "LR_TFIDF": LogisticRegression(max_iter=2000),
        "RF_TFIDF": RandomForestClassifier(n_jobs=-1),
        "SVM_TFIDF": CalibratedClassifierCV(LinearSVC(dual=False), cv=3),
    }

    models_emb = {
        "LR_EMB": LogisticRegression(max_iter=2000),
        "RF_EMB": RandomForestClassifier(n_jobs=-1),
        "SVM_EMB": CalibratedClassifierCV(LinearSVC(dual=False), cv=3),
    }

    models_fusion = {
        "LR_FUSION": LogisticRegression(max_iter=2000),
        "RF_FUSION": RandomForestClassifier(n_jobs=-1),
        "SVM_FUSION": CalibratedClassifierCV(LinearSVC(dual=False), cv=3),
    }

    # Run blocks
    train_eval_block(models_tfidf, X_train_tfidf, X_test_tfidf, y_train, y_test, y_test_bin, train_frac, "TF-IDF")
    train_eval_block(models_emb, X_train_emb, X_test_emb, y_train, y_test, y_test_bin, train_frac, "EMB")
    train_eval_block(models_fusion, X_train_fusion, X_test_fusion, y_train, y_test, y_test_bin, train_frac, "FUSION")

# ----------------------------
# Save results
# ----------------------------
results_df = pd.DataFrame(results)
results_df.to_csv(SAVE_RESULTS_PATH, index=False)

print("\n======================================")
print("ALL DONE ✅")
print("Saved results to:", SAVE_RESULTS_PATH)
if SAVE_CM_IMAGES:
    print("Saved confusion matrices to:", CM_DIR)
print("======================================\n")

results_df


Loaded: (38446, 3)
Classes: ['age' 'ethnicity' 'gender' 'not_cyberbullying' 'religion']

Class distribution:
cyberbullying_type
age                  7954
ethnicity            7847
religion             7698
gender               7570
not_cyberbullying    7377
Name: count, dtype: int64

 Sentence Embeddings Extraction

Computing embeddings for all texts (one-time)...


Batches:   0%|          | 0/1202 [00:00<?, ?it/s]

Embeddings shape: (38446, 384)

RUN SPLIT: 90_10

[TF-IDF] Fitting on training set and transforming...
[TF-IDF] Done. Shapes: (34601, 3000) (3845, 3000)

[EMB] Preparing embeddings split...
[EMB] Done. Shapes: (34601, 384) (3845, 384)

[FUSION] Building sparse fusion via hstack...
[FUSION] Done. Shapes: (34601, 3384) (3845, 3384)

🚀 Training LR_TFIDF on TF-IDF (90% train)...
✅ Done LR_TFIDF | Acc=0.9183 F1=0.9180

🚀 Training RF_TFIDF on TF-IDF (90% train)...
✅ Done RF_TFIDF | Acc=0.9225 F1=0.9223

🚀 Training SVM_TFIDF on TF-IDF (90% train)...
✅ Done SVM_TFIDF | Acc=0.9215 F1=0.9206

🚀 Training LR_EMB on EMB (90% train)...
✅ Done LR_EMB | Acc=0.9072 F1=0.9061

🚀 Training RF_EMB on EMB (90% train)...
✅ Done RF_EMB | Acc=0.8936 F1=0.8935

🚀 Training SVM_EMB on EMB (90% train)...
✅ Done SVM_EMB | Acc=0.9095 F1=0.9081

🚀 Training LR_FUSION on FUSION (90% train)...
✅ Done LR_FUSION | Acc=0.9264 F1=0.9257

🚀 Training RF_FUSION on FUSION (90% train)...
✅ Done RF_FUSION | Acc=0.9022 F1=0.9025



,model,train_frac,accuracy,precision,recall,f1,roc_auc
0,LR_TFIDF,0.9,0.918336,0.921234,0.917197,0.918046,0.987453
1,RF_TFIDF,0.9,0.922497,0.924861,0.921460,0.922267,0.987792
2,SVM_TFIDF,0.9,0.921456,0.922395,0.920209,0.920645,0.988256
3,LR_EMB,0.9,0.907152,0.907482,0.905806,0.906133,0.987188
4,RF_EMB,0.9,0.893628,0.898583,0.892539,0.893466,0.980953
5,SVM_EMB,0.9,0.909493,0.909368,0.907997,0.908061,0.985915
6,LR_FUSION,0.9,0.926398,0.927236,0.925314,0.925715,0.990633
7,RF_FUSION,0.9,0.902211,0.910122,0.901118,0.902457,0.985076
8,SVM_FUSION,0.9,0.924577,0.924971,0.923371,0.923731,0.990066
9,LR_TFIDF,0.8,0.915215,0.916142,0.913820,0.914346,0.987561


#=============================
# Tunning Part
#=============================

In [ ]:
# ==========================================================
# SCRIPT 5: FINAL MODEL TUNING & COMPARISON
# - Models: LR (Fusion) vs. SVM (Fusion) vs. BiLSTM
# - Split: 80% Training / 20% Held-out Test (Standardized)
# - Tuning: GridSearchCV (LR/SVM) / Simple validation loop (BiLSTM)
# ==========================================================

import os
import numpy as np
import pandas as pd
import joblib
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from scipy.sparse import hstack, csr_matrix

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import Sequential, layers, callbacks

warnings.filterwarnings("ignore")

# ----------------------------
# Config
# ----------------------------
DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/FYP/cyberbullying_cleaned_dropOverlappedRows.csv"
OUTPUT_DIR = "/content/drive/MyDrive/Colab Notebooks/FYP/final_tuned_models"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ----------------------------
# Helper: Save Confusion Matrix
# ----------------------------
def save_confusion_matrix(y_true, y_pred, labels, filename):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(filename)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, filename), dpi=200)
    plt.close()

def eval_and_print(name, y_true, y_pred, y_proba=None, classes=None):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

    roc = np.nan
    if y_proba is not None and classes is not None:
        y_true_bin = label_binarize(y_true, classes=list(range(len(classes))))
        roc = roc_auc_score(y_true_bin, y_proba, average="macro", multi_class="ovr")

    print(f"\n🏆 {name} (Locked Test Set)")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-macro : {f1:.4f}")
    if not np.isnan(roc):
        print(f"ROC-AUC  : {roc:.4f}")

    return acc, prec, rec, f1, roc

# ----------------------------
# 1. Load & Split Data (80/20 Standard)
# ----------------------------
print("Loading Data...")
df = pd.read_csv(DATA_PATH)

df["clean_text"] = df["clean_text"].fillna("").astype(str)
texts = df["clean_text"].tolist()
labels = df["cyberbullying_type"].astype(str).tolist()

le = LabelEncoder()
y = le.fit_transform(labels)
num_classes = len(le.classes_)
print(f"Classes: {le.classes_}")

print("\nPerforming Standardized 80/20 Split...")
X_train_text, X_test_text, y_train, y_test = train_test_split(
    texts, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)
print(f"Training Set: {len(X_train_text)} rows")
print(f"Test Set:     {len(X_test_text)} rows (LOCKED)")

# ==========================================================
# Shared Feature Engineering for LR/SVM (Fusion)
# Fit on TRAIN only (no leakage)
# ==========================================================
print("\n" + "="*50)
print(" FEATURE ENGINEERING (Fusion) for LR/SVM")
print("="*50)

print("Generating TF-IDF (fit on TRAIN only)...")
tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1,2), stop_words="english")
X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf  = tfidf.transform(X_test_text)

print("Generating Embeddings (MiniLM)...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
X_train_emb = embedder.encode(X_train_text, convert_to_numpy=True, show_progress_bar=True)
X_test_emb  = embedder.encode(X_test_text, convert_to_numpy=True, show_progress_bar=True)

X_train_fusion = hstack([X_train_tfidf, csr_matrix(X_train_emb)], format="csr")
X_test_fusion  = hstack([X_test_tfidf,  csr_matrix(X_test_emb)],  format="csr")

# Save shared preprocessors for web deployment
joblib.dump(tfidf, os.path.join(OUTPUT_DIR, "tfidf_vectorizer.pkl"))
joblib.dump(le, os.path.join(OUTPUT_DIR, "label_encoder.pkl"))
# NOTE: SentenceTransformer is usually reloaded by name in deployment, not joblib’d.
# Save the model name:
with open(os.path.join(OUTPUT_DIR, "embedding_model_name.txt"), "w") as f:
    f.write("all-MiniLM-L6-v2")

# ==========================================================
# PART A: TUNE Logistic Regression (FUSION)
# ==========================================================
print("\n" + "="*50)
print(" PART A: TUNING Logistic Regression (FUSION)")
print("="*50)

lr = LogisticRegression(max_iter=3000)  # keep simple; multinomial auto-handled in newer sklearn
param_grid_lr = {
    "C": [0.1, 0.5, 1, 2, 5],
    "solver": ["lbfgs"]
}

grid_lr = GridSearchCV(
    lr,
    param_grid_lr,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1
)

grid_lr.fit(X_train_fusion, y_train)
best_lr = grid_lr.best_estimator_
print("\n✅ Best LR params:", grid_lr.best_params_)
print("✅ Best CV F1:", f"{grid_lr.best_score_:.4f}")

# Evaluate on locked test
y_pred_lr = best_lr.predict(X_test_fusion)
y_proba_lr = best_lr.predict_proba(X_test_fusion)

lr_acc, lr_prec, lr_rec, lr_f1, lr_roc = eval_and_print(
    "FINAL LR (Fusion)", y_test, y_pred_lr, y_proba_lr, le.classes_
)

save_confusion_matrix(y_test, y_pred_lr, le.classes_, "CM_FINAL_LR.png")
joblib.dump(best_lr, os.path.join(OUTPUT_DIR, "final_lr_fusion.pkl"))

# ==========================================================
# PART B: TUNE SVM (FUSION)
# ==========================================================
print("\n" + "="*50)
print(" PART B: TUNING SVM (FUSION)")
print("="*50)

param_grid_svm = {"C": [0.1, 0.5, 1, 2, 5]}
svm_base = LinearSVC(dual=False, random_state=SEED)

grid_svm = GridSearchCV(
    svm_base,
    param_grid_svm,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1
)

grid_svm.fit(X_train_fusion, y_train)
best_C = grid_svm.best_params_["C"]
print(f"\n✅ Best SVM Params: C={best_C}")
print(f"✅ Best CV F1 Score: {grid_svm.best_score_:.4f}")

# Train final SVM on full training set
final_svm_base = LinearSVC(C=best_C, dual=False, random_state=SEED)

# Calibration to get probabilities for ROC-AUC / web confidence
# Use cv=3 to reduce time; still calibration is only on TRAIN
final_svm = CalibratedClassifierCV(final_svm_base, cv=3)
final_svm.fit(X_train_fusion, y_train)

# Evaluate on locked test
y_pred_svm = final_svm.predict(X_test_fusion)
y_proba_svm = final_svm.predict_proba(X_test_fusion)

svm_acc, svm_prec, svm_rec, svm_f1, svm_roc = eval_and_print(
    "FINAL SVM (Fusion)", y_test, y_pred_svm, y_proba_svm, le.classes_
)

save_confusion_matrix(y_test, y_pred_svm, le.classes_, "CM_FINAL_SVM.png")
joblib.dump(final_svm, os.path.join(OUTPUT_DIR, "final_svm_fusion.pkl"))

# ==========================================================
# PART C: TUNE BiLSTM (token sequences)
# ==========================================================
print("\n" + "="*50)
print(" PART C: TUNING BiLSTM")
print("="*50)

MAX_WORDS = 20000
MAX_LEN = 100

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq  = tokenizer.texts_to_sequences(X_test_text)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN)
X_test_pad  = pad_sequences(X_test_seq, maxlen=MAX_LEN)

hyperparams = [
    {"units": 64,  "dropout": 0.2},
    {"units": 128, "dropout": 0.3},
]

best_bilstm_model = None
best_val_acc = -1
best_params = None

for params in hyperparams:
    u = params["units"]
    d = params["dropout"]
    print(f"\nTesting BiLSTM: Units={u}, Dropout={d}...")

    model = Sequential([
        layers.Embedding(MAX_WORDS, 64, input_length=MAX_LEN),
        layers.Bidirectional(layers.LSTM(u, return_sequences=False)),
        layers.Dropout(d),
        layers.Dense(64, activation="relu"),
        layers.Dense(num_classes, activation="softmax")
    ])

    model.compile(
        loss="sparse_categorical_crossentropy",
        optimizer="adam",
        metrics=["accuracy"]
    )

    hist = model.fit(
        X_train_pad, y_train,
        epochs=6,
        batch_size=128,
        validation_split=0.2,
        callbacks=[callbacks.EarlyStopping(patience=1, restore_best_weights=True)],
        verbose=1
    )

    val_acc = max(hist.history["val_accuracy"])
    print(f"-> Val Accuracy: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_bilstm_model = model
        best_params = params

print(f"\n✅ Best BiLSTM Params: {best_params}")

# Evaluate on locked test
y_proba_bilstm = best_bilstm_model.predict(X_test_pad)
y_pred_bilstm = np.argmax(y_proba_bilstm, axis=1)

bilstm_acc, bilstm_prec, bilstm_rec, bilstm_f1, bilstm_roc = eval_and_print(
    "FINAL BiLSTM", y_test, y_pred_bilstm, y_proba_bilstm, le.classes_
)

save_confusion_matrix(y_test, y_pred_bilstm, le.classes_, "CM_FINAL_BiLSTM.png")

# Save BiLSTM + tokenizer
best_bilstm_model.save(os.path.join(OUTPUT_DIR, "final_bilstm.keras"))
joblib.dump(tokenizer, os.path.join(OUTPUT_DIR, "tokenizer.pkl"))

# ==========================================================
# FINAL SUMMARY
# ==========================================================
print("\n" + "="*50)
print(" FINAL COMPARISON (Locked Test Set)")
print("="*50)

results_df = pd.DataFrame([
    {
        "Model": "LR (Fusion)",
        "Accuracy": lr_acc,
        "Precision(macro)": lr_prec,
        "Recall(macro)": lr_rec,
        "F1-Macro": lr_f1,
        "ROC-AUC(macro)": lr_roc,
        "Best Params": str(grid_lr.best_params_)
    },
    {
        "Model": "SVM (Fusion)",
        "Accuracy": svm_acc,
        "Precision(macro)": svm_prec,
        "Recall(macro)": svm_rec,
        "F1-Macro": svm_f1,
        "ROC-AUC(macro)": svm_roc,
        "Best Params": f"C={best_C} (LinearSVC + Calibrated)"
    },
    {
        "Model": "BiLSTM",
        "Accuracy": bilstm_acc,
        "Precision(macro)": bilstm_prec,
        "Recall(macro)": bilstm_rec,
        "F1-Macro": bilstm_f1,
        "ROC-AUC(macro)": bilstm_roc,
        "Best Params": str(best_params)
    }
])

print(results_df.to_string(index=False))
results_df.to_csv(os.path.join(OUTPUT_DIR, "final_comparison_results.csv"), index=False)

print(f"\nAll models and results saved to: {OUTPUT_DIR}")
print("Saved artifacts include:")
print("- final_lr_fusion.pkl")
print("- final_svm_fusion.pkl")
print("- tfidf_vectorizer.pkl")
print("- label_encoder.pkl")
print("- embedding_model_name.txt")
print("- final_bilstm.keras")
print("- tokenizer.pkl")
print("- confusion matrices (CM_FINAL_*.png)")


Loading Data...
Classes: ['age' 'ethnicity' 'gender' 'not_cyberbullying' 'religion']

Performing Standardized 80/20 Split...
Training Set: 30756 rows
Test Set:     7690 rows (LOCKED)

 FEATURE ENGINEERING (Fusion) for LR/SVM
Generating TF-IDF (fit on TRAIN only)...
Generating Embeddings (MiniLM)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/962 [00:00<?, ?it/s]

Batches:   0%|          | 0/241 [00:00<?, ?it/s]


 PART A: TUNING Logistic Regression (FUSION)
Fitting 5 folds for each of 5 candidates, totalling 25 fits

✅ Best LR params: {'C': 2, 'solver': 'lbfgs'}
✅ Best CV F1: 0.9258

🏆 FINAL LR (Fusion) (Locked Test Set)
Accuracy : 0.9243
Precision: 0.9233
Recall   : 0.9229
F1-macro : 0.9229
ROC-AUC  : 0.9908

 PART B: TUNING SVM (FUSION)
Fitting 5 folds for each of 5 candidates, totalling 25 fits

✅ Best SVM Params: C=0.1
✅ Best CV F1 Score: 0.9274

🏆 FINAL SVM (Fusion) (Locked Test Set)
Accuracy : 0.9261
Precision: 0.9248
Recall   : 0.9246
F1-macro : 0.9245
ROC-AUC  : 0.9902

 PART C: TUNING BiLSTM

Testing BiLSTM: Units=64, Dropout=0.2...
Epoch 1/6
193/193 ━━━━━━━━━━━━━━━━━━━━ 62s 300ms/step - accuracy: 0.6155 - loss: 0.9914 - val_accuracy: 0.9200 - val_loss: 0.2224
Epoch 2/6
193/193 ━━━━━━━━━━━━━━━━━━━━ 97s 380ms/step - accuracy: 0.9311 - loss: 0.2030 - val_accuracy: 0.9252 - val_loss: 0.2218
Epoch 3/6
193/193 ━━━━━━━━━━━━━━━━━━━━ 58s 300ms/step - accuracy: 0.9552 - loss: 0.1376 - val_accu